# Combined Opportunity Report (Product + Market)

This notebook combines the product-level and market-level opportunity outputs and presents a single view with readable country and product names. It also builds decision examples from the highest-confidence market opportunities.


## Objectives

- Load the latest product and market ranking outputs.
- Map country ISO3 codes to full country names using World Bank metadata.
- Resolve product names using available HS descriptions from the market output (fallback to code-based labels when missing).
- Summarize top product opportunities, top market opportunities, and produce decision examples.

In [100]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## Load output files

The notebook looks for outputs in the standard local results folder and the notebook-local results folder. This keeps the workflow compatible with both local and notebook-relative paths.

In [101]:
BASE_RESULTS_DIRS = [
    Path("results"),
    Path("notebooks/classification/results"),
]

def find_result_file(filename: str) -> Path:
    for base_dir in BASE_RESULTS_DIRS:
        candidate = base_dir / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename} in any known results folder.")

product_path = find_result_file("product_opportunity_ranking.csv")
market_path = find_result_file("market_opportunity_ranking.csv")

product_df = pd.read_csv(product_path)
market_df = pd.read_csv(market_path)

print("Product file:", product_path)
print("Market file:", market_path)
print("Product rows:", product_df.shape)
print("Market rows:", market_df.shape)

Product file: results\product_opportunity_ranking.csv
Market file: results\market_opportunity_ranking.csv
Product rows: (1575, 14)
Market rows: (596272, 22)


## Validate expected columns

This quick check ensures the core columns required for reporting are present before proceeding.

In [102]:
required_product_cols = {
    "label_year",
    "feature_year",
    "Exporter_ISO3",
    "Product",
    "opp_score_T",
    "pred_label",
    "prob_High",
}

required_market_cols = {
    "label_year",
    "feature_year",
    "Importer_ISO3",
    "Product",
    "Importer_Demand_T",
    "Algeria_Share_in_Importer_T",
    "pred_label",
    "prob_High",
}

missing_product = required_product_cols - set(product_df.columns)
missing_market = required_market_cols - set(market_df.columns)

if missing_product:
    raise ValueError(f"Missing product columns: {sorted(missing_product)}")
if missing_market:
    raise ValueError(f"Missing market columns: {sorted(missing_market)}")

print("Required columns are present.")

Required columns are present.


## Country name mapping

World Bank metadata provides a consistent ISO3-to-country-name mapping. The mapping is applied to exporters and importers for readable reporting.

In [103]:
country_meta_path = Path("../../data/raw/worldbank/gdp-current-usd/Metadata_Country_API_NY.GDP.MKTP.CD_DS2_en_csv_v2_133326.csv")
country_meta = pd.read_csv(country_meta_path)

country_lookup = (
    country_meta[["Country Code", "TableName"]]
    .dropna()
    .drop_duplicates()
    .set_index("Country Code")["TableName"]
    .to_dict()
)

product_df["Exporter_Name"] = product_df["Exporter_ISO3"].map(country_lookup)
market_df["Importer_Name"] = market_df["Importer_ISO3"].map(country_lookup)

print("Example exporter names:")
display(product_df[["Exporter_ISO3", "Exporter_Name"]].drop_duplicates().head())
print("Example importer names:")
display(market_df[["Importer_ISO3", "Importer_Name"]].drop_duplicates().head())

Example exporter names:


,Exporter_ISO3,Exporter_Name
0,DZA,Algeria


Example importer names:


,Importer_ISO3,Importer_Name
0,DEU,Germany
2,USA,United States
6,FRA,France
10,BEL,Belgium
21,AUT,Austria


## HS version detection


### How the HS reference is used

This step reads the HS reference tables and builds a lookup from HS6 code to description. The lookup is then used later to replace numeric product codes with human-readable names. If multiple HS versions are available, the version with the highest coverage of your HS6 codes is selected.

In [104]:
HS_REFERENCE_DIR = Path("../../data/raw/hs_reference")
HS_VERSION_FILES = {
    "HS1992": HS_REFERENCE_DIR / "hs1992.csv",
    "HS1996": HS_REFERENCE_DIR / "hs1996.csv",
    "HS2002": HS_REFERENCE_DIR / "hs2002.csv",
    "HS2007": HS_REFERENCE_DIR / "hs2007.csv",
    "HS2012": HS_REFERENCE_DIR / "hs2012.csv",
    "HS2017": HS_REFERENCE_DIR / "hs2017.csv",
    "HS2022": HS_REFERENCE_DIR / "hs2022.csv",
}


def read_csv_flexible(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "latin1"]
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def normalize_code_series(series: pd.Series) -> pd.Series:
    codes = series.astype(str).str.replace(r"\D", "", regex=True)
    codes = codes.where(codes.str.len() > 0)
    codes = codes.str[:6]
    codes = codes.where(codes.str.len() == 6)
    return codes


def pick_column_by_name(df: pd.DataFrame, keywords: tuple[str, ...]) -> str | None:
    lowered = {col: col.lower() for col in df.columns}
    for col, col_lower in lowered.items():
        if any(key in col_lower for key in keywords):
            return col
    return None


def find_code_column(df: pd.DataFrame) -> str | None:
    name_hint = pick_column_by_name(df, ("code", "hs", "product code"))
    if name_hint:
        return name_hint

    best_col = None
    best_score = 0.0
    for col in df.columns:
        score = normalize_code_series(df[col]).notna().mean()
        if score > best_score:
            best_col = col
            best_score = score
    return best_col


def is_unit_like(series: pd.Series) -> bool:
    text = series.astype(str).str.strip().str.lower()
    top_value = text.value_counts(dropna=True).head(1)
    if top_value.empty:
        return False
    value = top_value.index[0]
    freq = float(top_value.iloc[0]) / max(len(text), 1)
    unit_values = {"kg", "kilogram", "ton", "tons", "t", "mt", "usd", "us$", "unit"}
    return freq > 0.6 and (value in unit_values or len(value) <= 4)


def description_score(series: pd.Series) -> float:
    text = series.astype(str)
    alpha_ratio = text.str.contains(r"[A-Za-z]", regex=True).mean()
    avg_len = text.str.len().mean()
    return alpha_ratio * avg_len


def find_desc_column(df: pd.DataFrame, code_col: str | None) -> str | None:
    name_hint = pick_column_by_name(df, ("description", "desc", "name", "product"))
    if name_hint and name_hint != code_col:
        if not is_unit_like(df[name_hint]):
            return name_hint

    best_col = None
    best_score = 0.0
    for col in df.columns:
        if col == code_col:
            continue
        if df[col].dtype == object:
            if is_unit_like(df[col]):
                continue
            score = description_score(df[col])
            if score > best_score:
                best_col = col
                best_score = score
    return best_col


def clean_description(series: pd.Series) -> pd.Series:
    text = series.astype(str)
    text = text.str.replace(r"^\s*\d{2,6}\s*-\s*", "", regex=True)
    return text


def build_reference_lookup(df: pd.DataFrame) -> tuple[dict, str | None, str | None]:
    if "id" in df.columns:
        id_text = df["id"].astype(str)
        if id_text.str.contains(r"-", regex=True).any():
            codes = normalize_code_series(id_text)
            descs = clean_description(id_text)
            lookup = (
                pd.DataFrame({"code": codes, "desc": descs})
                .dropna()
                .drop_duplicates(subset=["code"])
                .set_index("code")["desc"]
                .to_dict()
            )
            return lookup, "id", "id"

    code_col = find_code_column(df)
    desc_col = find_desc_column(df, code_col)
    if not code_col or not desc_col:
        return {}, code_col, desc_col
    codes = normalize_code_series(df[code_col])
    descs = clean_description(df[desc_col])
    lookup = (
        pd.DataFrame({"code": codes, "desc": descs})
        .dropna()
        .drop_duplicates(subset=["code"])
        .set_index("code")["desc"]
        .to_dict()
    )
    return lookup, code_col, desc_col


product_codes = normalize_code_series(product_df["Product"]).dropna().unique().tolist()
coverage_rows = []
hs_reference_lookup = {}

for version, path in HS_VERSION_FILES.items():
    if not path.exists():
        continue
    reference_df = read_csv_flexible(path)
    reference_lookup, code_col, desc_col = build_reference_lookup(reference_df)
    if not reference_lookup:
        print(f"{version}: could not infer code/description columns from {path.name}")
        print(f"Columns: {list(reference_df.columns)}")
        continue
    reference_codes = set(reference_lookup.keys())
    coverage = float(np.mean([code in reference_codes for code in product_codes]))
    coverage_rows.append({
        "version": version,
        "coverage": coverage,
        "path": str(path),
        "code_col": code_col,
        "desc_col": desc_col,
    })

if coverage_rows:
    coverage_table = pd.DataFrame(coverage_rows).sort_values("coverage", ascending=False)
    display(coverage_table)
    best_version = coverage_table.iloc[0]["version"]
    best_path = Path(coverage_table.iloc[0]["path"])
    hs_reference_lookup, _, _ = build_reference_lookup(read_csv_flexible(best_path))
    print(f"Selected HS version: {best_version}")
else:
    print("No HS reference files found. Add CSVs under data/raw/hs_reference to enable detection.")

,version,coverage,path,code_col,desc_col
0,HS2012,0.999299,..\..\data\raw\hs_reference\hs2012.csv,id,id
1,HS2017,0.990182,..\..\data\raw\hs_reference\hs2017.csv,id,id
2,HS2022,0.955820,..\..\data\raw\hs_reference\hs2022.csv,id,id


Selected HS version: HS2012


## Product naming

The market output contains HS6 descriptions. These are used to name products wherever possible. When a description is missing, the report falls back to a code-based label (for example, "HS6 080810").

In [105]:
def normalize_hs_code(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    digits = "".join(ch for ch in text if ch.isdigit())
    if not digits:
        return None
    if len(digits) < 6:
        digits = digits.zfill(6)
    return digits


product_df["Product_Code"] = product_df["Product"].apply(normalize_hs_code)
market_df["Product_Code"] = market_df["Product"].apply(normalize_hs_code)

product_lookup = {}
if "hs_reference_lookup" in globals() and hs_reference_lookup:
    product_lookup.update(hs_reference_lookup)

if "HS6_Description" in market_df.columns:
    market_lookup = (
        market_df[["Product_Code", "HS6_Description"]]
        .dropna()
        .drop_duplicates()
        .set_index("Product_Code")["HS6_Description"]
        .to_dict()
    )
    for code, desc in market_lookup.items():
        product_lookup.setdefault(code, desc)


def resolve_product_name(code: str | None) -> str:
    if not code:
        return "Unknown product"
    if code in product_lookup:
        return product_lookup[code]
    return f"HS6 {code}"


product_df["Product_Name"] = product_df["Product_Code"].apply(resolve_product_name)
market_df["Product_Name"] = market_df["Product_Code"].apply(resolve_product_name)

display(product_df[["Product", "Product_Code", "Product_Name"]].head())

,Product,Product_Code,Product_Name
0,253090,253090,-Other
1,210690,210690,-Other
2,80810,080810,-Apples
3,210500,210500,"Ice cream and other edible ice, whether or not..."
4,151110,151110,-Crude oil


## Product opportunity summary

This section highlights the strongest product opportunities based on predicted high class probability and the opportunity score.

In [106]:
top_products = (
    product_df
    .sort_values(["prob_High", "opp_score_T"], ascending=False)
    .head(20)
    [["Product_Code", "Product_Name", "opp_score_T", "prob_High", "Global_Demand_T"]]
)

display(top_products)

,Product_Code,Product_Name,opp_score_T,prob_High,Global_Demand_T
0,253090,-Other,0.609811,0.999995,10137005.0
1,210690,-Other,0.635505,0.999993,57266740.0
2,080810,-Apples,0.656040,0.999991,7354612.5
3,210500,"Ice cream and other edible ice, whether or not...",0.614632,0.999991,5424219.0
4,151110,-Crude oil,0.653971,0.999989,16462242.0
5,080550,"-Lemons (Citrus limon, Citrus limonum) and lim...",0.590136,0.999985,4060181.8
6,070200,"Tomatoes, fresh or chilled.",0.618985,0.999983,10425973.0
7,080610,-Fresh,0.626289,0.999979,9836397.0
8,681099,-- Other,0.635956,0.999975,6229779.0
9,080440,-Avocados,0.653807,0.999968,7672133.0


## Market opportunity summary

Market opportunities are filtered to the highest-confidence predictions and ranked by probability and importer demand.

In [107]:
top_markets = (
    market_df[market_df["pred_label"] == "High"]
    .sort_values(["prob_High", "Importer_Demand_T"], ascending=False)
    .head(20)
    [[
        "Importer_Name",
        "Importer_ISO3",
        "Product_Code",
        "Product_Name",
        "Importer_Demand_T",
        "Algeria_Share_in_Importer_T",
        "prob_High",
    ]]
)

display(top_markets)

,Importer_Name,Importer_ISO3,Product_Code,Product_Name,Importer_Demand_T,Algeria_Share_in_Importer_T,prob_High
0,Germany,DEU,081110,-Strawberries,148600.810,0.0,0.999886
1,Germany,DEU,071090,-Mixtures of vegetables,111339.290,0.0,0.999868
2,United States,USA,680520,-On a base of paper or paperboard only,149443.800,0.0,0.999864
3,Germany,DEU,210500,"Ice cream and other edible ice, whether or not...",492931.200,0.0,0.999860
4,United States,USA,070960,-Fruits of the genus Capsicum or of the genus ...,2031307.600,0.0,0.999858
5,Germany,DEU,080390,-Other,848582.440,0.0,0.999857
6,France,FRA,190540,"-Rusks, toasted bread and similar toasted prod...",76277.445,0.0,0.999857
7,Germany,DEU,080212,-- Shelled,560627.440,0.0,0.999853
8,United States,USA,071080,-Other vegetables,874840.700,0.0,0.999853
9,Germany,DEU,190220,"-Stuffed pasta, whether or not cooked or other...",238790.640,0.0,0.999852


## Decision examples (Top 5)

Decision examples connect high-probability market opportunities with top product priorities. These are not recommendations; they illustrate how the combined outputs can be interpreted in a decision workflow.

### Decision logic

The examples are created by filtering to predicted `High` market opportunities, then ranking by `prob_High` (model confidence) and `Importer_Demand_T` (market size). The script first tries to keep only products that also appear in the top product list; if none match, it falls back to the full set of high-confidence markets. The final text highlights demand size and whether Algeria already has a non-trivial share (>= 2%) to distinguish expansion versus new-entry cases.

In [110]:
top_product_codes = set(top_products["Product_Code"].dropna().tolist())

candidate_examples = market_df[market_df["pred_label"] == "High"]
candidate_examples = candidate_examples[candidate_examples["Product_Code"].isin(top_product_codes)]

if candidate_examples.empty:
    candidate_examples = market_df[market_df["pred_label"] == "High"]

candidate_examples = candidate_examples.sort_values(["prob_High", "Importer_Demand_T"], ascending=False)
decision_examples = candidate_examples.head(5).copy()

def build_decision_example(row: pd.Series) -> str:
    importer = row.get("Importer_Name") or row.get("Importer_ISO3", "Unknown")
    product = row.get("Product_Name") or row.get("Product_Code", "Unknown product")
    demand = row.get("Importer_Demand_T")
    share = row.get("Algeria_Share_in_Importer_T")

    demand_text = "high demand"
    if pd.notna(demand):
        demand_text = f"demand {demand:,.0f}"

    share_text = "low share"
    if pd.notna(share) and share >= 0.02:
        share_text = "existing share"

    return f"Target {importer} for {product} due to {demand_text} and {share_text}."

decision_examples["Decision_Example"] = decision_examples.apply(build_decision_example, axis=1)

example_view = decision_examples[[
    "Importer_Name",
    "Importer_ISO3",
    "Product_Code",
    "Product_Name",
    "Importer_Demand_T",
    "Algeria_Share_in_Importer_T",
    "prob_High",
    "Decision_Example",
]]

display(example_view)

,Importer_Name,Importer_ISO3,Product_Code,Product_Name,Importer_Demand_T,Algeria_Share_in_Importer_T,prob_High,Decision_Example
3,Germany,DEU,210500,"Ice cream and other edible ice, whether or not...",492931.200,0.0,0.999860,Target Germany for Ice cream and other edible ...
4,United States,USA,070960,-Fruits of the genus Capsicum or of the genus ...,2031307.600,0.0,0.999858,Target United States for -Fruits of the genus ...
12,Germany,DEU,080430,-Pineapples,78951.375,0.0,0.999849,Target Germany for -Pineapples due to demand 7...
14,Germany,DEU,070960,-Fruits of the genus Capsicum or of the genus ...,914591.000,0.0,0.999847,Target Germany for -Fruits of the genus Capsic...
25,Germany,DEU,070200,"Tomatoes, fresh or chilled.",1546312.000,0.0,0.999840,"Target Germany for Tomatoes, fresh or chilled...."


## Save combined decision examples

The combined examples are saved next to the source outputs for easy reuse in dashboards or reports.

In [109]:
output_dir = product_path.parent
output_path = output_dir / "combined_decision_examples.csv"
example_view.to_csv(output_path, index=False)

print("Saved combined decision examples to:", output_path)

Saved combined decision examples to: results\combined_decision_examples.csv


## Interpretation notes

- Country names are sourced from World Bank metadata and may include regional aggregates.
- Product names rely on HS6 descriptions present in the market output. Missing descriptions fall back to the HS6 code label.
- Decision examples prioritize high-confidence predictions and are intended for human review.

## Conclusion

The combined view aligns product-level opportunity scores with market-level demand signals and adds readable country and product names for reporting. With the HS lookup in place, the outputs are ready for decision review, prioritization, and downstream dashboards or reports.